# CNN Model Creation and Training.

In [ ]:
# !pip install tensorflow

In [ ]:
# imports 
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import pickle as pk
import matplotlib.pyplot as plt

In [ ]:
TRAIN_TEST_PATH = "TRAIN_TEST"


In [ ]:
# load X_train,X_test,Y_train,Y_test.

In [ ]:
X_train = np.load(os.path.join(TRAIN_TEST_PATH,"X_train.npy"))
X_test = np.load(os.path.join(TRAIN_TEST_PATH,"X_test.npy"))
Y_train = np.load(os.path.join(TRAIN_TEST_PATH,"Y_train.npy"))
Y_test = np.load(os.path.join(TRAIN_TEST_PATH,"Y_test.npy"))

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

In [ ]:
X_train = X_train.transpose(0, 2, 3, 1)
X_test = X_test.transpose(0, 2, 3, 1)

In [ ]:
# X_train = (X_train + 80) / 80
# X_test = (X_test + 80) / 80
# Normalization if needed.

In [ ]:
print(np.unique(Y_train,return_counts=True))
print(np.unique(Y_test,return_counts=True))

In [ ]:
print(X_train.max())
print(X_train.min())

In [ ]:
def create_cnn_model(input_shape=(128,216,1)):
    model = models.Sequential()

    model.add(layers.Conv2D(32,(3,3),padding='same',input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(64,(3,3),padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(128,(3,3),padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.MaxPooling2D((2,2)))


    # model.add(layers.Flatten())777777
    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(64))
    model.add(layers.ReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Dense(1,activation='sigmoid'))

    return model


In [ ]:
threat_detector = create_cnn_model()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
threat_detector.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
threat_detector.summary()

In [ ]:
history = threat_detector.fit(
    X_train,
    Y_train,
    validation_data=(X_test,Y_test),
    epochs=50,
    batch_size=16,
    callbacks=[early_stop],
    verbose = 1
)

In [ ]:
best_epoch = np.argmax(history.history['val_accuracy']) + 1

print("Best Epoch:", best_epoch)
print("Best Validation Accuracy:",
      max(history.history['val_accuracy']))

In [ ]:
loss,acc = threat_detector.evaluate(X_test,Y_test)

In [ ]:
print("Loss :",loss)
print("Accuracy :",acc)

# Accuracy : .875 and loss 0.31602 
# with out Normalization.
# and with normalilzation 
# Accuracy : .6858
# Loss = 0.547
# So no normalization

In [ ]:
threat_detector.summary()

In [ ]:
Y_pred_prob = threat_detector.predict(X_test)

Y_pred = (Y_pred_prob > 0.5).astype(int)


print(Y_pred[:10])

In [ ]:
Y_pred = Y_pred.flatten()
Y_test = Y_test.flatten()

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("Confusion Matrix")
print(confusion_matrix(Y_test, Y_pred))

print("\nClassification Report")
print(classification_report(Y_test, Y_pred))

print("\nPrecision :", precision_score(Y_test, Y_pred))
print("Recall    :", recall_score(Y_test, Y_pred))
print("F1 Score  :", f1_score(Y_test, Y_pred))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(Y_test, Y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Non Threat','Threat'],
    yticklabels=['Non Threat','Threat']
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(history.history['accuracy'],label='Train Accuracy')
plt.plot(history.history['val_accuracy'],label='Validation Accuracy')
plt.legend()
plt.title("Accuracy Curve")
plt.show()

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(history.history['loss'],label='Train Loss')
plt.plot(history.history['val_loss'],label='Validation Loss')
plt.legend()
plt.title("Loss Curve")
plt.show()

In [ ]:
# threat_detector.save("../models/threat_detector.keras")
# print("Model Saved Succesfully!!!")

In [ ]:
# with open("../models/training_history.pkl","wb") as f:
#     pk.dump(history.history,f)